# Stage 3 — Fine-tuning SHViT on UCF101

This notebook fine-tunes pretrained SHViT models (S1..S4) on UCF101
using `finetune_shvit_ucf101.py`. Data preprocessing follows
Tip-Adapter (`splits.py`, CLIP normalization, bicubic resize).

**Before running:** `Runtime → Change runtime type → T4 GPU`

### What this notebook does
1. Verifies GPU, mounts Drive (optional), prepares paths
2. Clones SHViT + this project repo, installs deps
3. Downloads SHViT pretrained weights (S1..S4)
4. Runs `finetune_shvit_ucf101.py` for each variant
5. Plots training curves

All outputs land under `<OUT_ROOT>/Stage 3: fine-tuning SHViT/<variant>/`.


## 0. GPU check

In [ ]:
import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))

## 1. (Optional) Mount Drive

Persist the dataset, weights, and checkpoints across sessions.

In [ ]:
USE_DRIVE = True

OUT_ROOT = '/content/CV_Research_Paper_UCF101'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT   = '/content/drive/MyDrive/ucf101_data'
    WEIGHTS_DIR = '/content/drive/MyDrive/shvit_weights'
    OUT_ROOT    = '/content/drive/MyDrive/CV_Research_Paper_UCF101'
else:
    DATA_ROOT   = '/content/ucf101_data'
    WEIGHTS_DIR = '/content/weights'

OUTPUT_DIR = f'{OUT_ROOT}/Stage 3: fine-tuning SHViT'
WEIGHTS_PATH = f'{WEIGHTS_DIR}/shvit_s4.pth'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)

print('Data       :', DATA_ROOT)
print('Weights    :', WEIGHTS_PATH)
print('Output root:', OUT_ROOT)
print('Output dir :', OUTPUT_DIR)


In [ ]:
# Optional: copy the dataset to local SSD for faster I/O.
import shutil, os, time
src = '/content/drive/MyDrive/ucf101_data'
dst = '/content/ucf101_local'

if USE_DRIVE and not os.path.exists(dst) and os.path.isdir(src):
    print('Copying dataset to local SSD...')
    t0 = time.time()
    shutil.copytree(src, dst)
    print(f'done in {time.time()-t0:.0f}s')
    DATA_ROOT = dst
elif os.path.exists(dst):
    DATA_ROOT = dst
print('DATA_ROOT now:', DATA_ROOT)


## 2. Clone repos and install dependencies

In [ ]:
import os, shutil

if not os.path.isdir('/content/SHViT'):
    !git clone https://github.com/ysj9909/SHViT.git /content/SHViT

REPO = '/content/Vision_Project_spring_26'
if not os.path.isdir(REPO):
    !git clone -b Vision_Project_spring_26_UCF101 \
        https://github.com/saif-farid-tech/Vision_Project_spring_26.git {REPO}

for fname in [
    'Stage 3: fine-tuning SHViT/finetune_shvit_ucf101.py',
    'splits.py',
    'metrics.py',
    'augmentation.py',
    'prepare_ucf101.py',
]:
    shutil.copy(f'{REPO}/{fname}', f'/content/{os.path.basename(fname)}')
    print('Copied:', os.path.basename(fname))

DATASETS_DST = '/content/datasets'
if os.path.isdir(DATASETS_DST):
    shutil.rmtree(DATASETS_DST)
shutil.copytree(f'{REPO}/datasets', DATASETS_DST)
print('Vendored datasets/ package.')


In [ ]:
# Install SHViT deps (timm pinned, --no-deps so torch isn't downgraded).
# gdown is used by prepare_ucf101.py to fetch the CoOp split.
!pip install -q timm==0.5.4 --no-deps
!pip install -q einops==0.4.1 easydict gdown
print('Dependencies installed.')


## 3. Download SHViT-S4 pretrained weights

In [ ]:
os.makedirs(WEIGHTS_DIR, exist_ok=True)
if not os.path.exists(WEIGHTS_PATH):
    !wget -q --show-progress \
        https://github.com/ysj9909/SHViT/releases/download/v1.0/shvit_s4.pth \
        -O {WEIGHTS_PATH}
size_mb = os.path.getsize(WEIGHTS_PATH) / 1e6
print(f'Checkpoint: {size_mb:.1f} MB  ->  {WEIGHTS_PATH}')

# Also pre-download the UCF101 split + images so the first training run is fast
!python /content/prepare_ucf101.py --root {DATA_ROOT}


## 4. Fine-tune SHViT on UCF101

UCF101 (mid-frame form) is moderate (~1.4 GB unpacked, ~13,320 images
across 101 action classes). Each epoch on a T4 takes ~30 sec with
batch 64 — 30 epochs ≈ 15 min per SHViT variant.


In [ ]:
# Quick smoke-test: 2 epochs to verify everything runs
!python /content/finetune_shvit_ucf101.py \
    --shvit-dir  /content/SHViT \
    --finetune   {WEIGHTS_PATH} \
    --data-root  {DATA_ROOT} \
    --output-dir "{OUTPUT_DIR}/shvit_s4_smoke" \
    --epochs 2 \
    --batch-size 64 \
    --lr 1e-4 \
    --num-workers 8


In [ ]:
# Train all 4 SHViT variants sequentially.
MODELS = ['shvit_s4', 'shvit_s3', 'shvit_s2', 'shvit_s1']

# Per-variant weight decay from the SHViT paper (Table 7).
WEIGHT_DECAY = {
    'shvit_s1': 0.025,
    'shvit_s2': 0.032,
    'shvit_s3': 0.035,
    'shvit_s4': 0.050,
}

for model_name in MODELS:
    model_output_dir = f'{OUTPUT_DIR}/{model_name}'
    weights_path = f'{WEIGHTS_DIR}/{model_name}.pth'

    if not os.path.exists(weights_path):
        print(f'\n=== Downloading {model_name} pretrained weights ===')
        !wget -q --show-progress \
            https://github.com/ysj9909/SHViT/releases/download/v1.0/{model_name}.pth \
            -O {weights_path}

    print(f'\n{"="*60}')
    print(f'Training {model_name}  (wd={WEIGHT_DECAY[model_name]})')
    print(f'{"="*60}')

    !python /content/finetune_shvit_ucf101.py \
        --model      {model_name} \
        --shvit-dir  /content/SHViT \
        --finetune   {weights_path} \
        --data-root  {DATA_ROOT} \
        --output-dir "{model_output_dir}" \
        --epochs 30 \
        --batch-size 64 \
        --lr 1e-4 \
        --warmup-epochs 5 \
        --weight-decay {WEIGHT_DECAY[model_name]} \
        --clip-grad 0.02 \
        --save-freq 10 \
        --num-workers 8

    print(f'\n=== Finished {model_name} ===\n')

print('All 4 SHViT variants done.')


In [ ]:
# Resume from a checkpoint if the session was interrupted
# !python /content/finetune_shvit_ucf101.py \
#     --shvit-dir  /content/SHViT \
#     --data-root  {DATA_ROOT} \
#     --output-dir "{OUTPUT_DIR}/shvit_s4" \
#     --resume     "{OUTPUT_DIR}/shvit_s4/checkpoint_010.pth" \
#     --epochs 30 \
#     --batch-size 64 \
#     --lr 1e-4 \
#     --num-workers 8


In [ ]:
# Re-download a corrupted pretrained weight if needed
import os, urllib.request

VARIANT = 'shvit_s3'
broken_path = f'{WEIGHTS_DIR}/{VARIANT}.pth'

if os.path.exists(broken_path):
    print(f'before: {os.path.getsize(broken_path)/1e6:.2f} MB')
    os.remove(broken_path)
    print('deleted broken file')

url = f'https://github.com/ysj9909/SHViT/releases/download/v1.0/{VARIANT}.pth'
print(f'downloading from {url}...')
urllib.request.urlretrieve(url, broken_path)
print(f'after:  {os.path.getsize(broken_path)/1e6:.2f} MB')


In [ ]:
# Per-variant retry helper
MODELS_TO_REDO = ['shvit_s3']

for model_name in MODELS_TO_REDO:
    weights_path = f'{WEIGHTS_DIR}/{model_name}.pth'
    model_output_dir = f'{OUTPUT_DIR}/{model_name}'

    !python /content/finetune_shvit_ucf101.py \
        --model      {model_name} \
        --shvit-dir  /content/SHViT \
        --finetune   {weights_path} \
        --data-root  {DATA_ROOT} \
        --output-dir "{model_output_dir}" \
        --epochs 30 \
        --batch-size 64 \
        --lr 1e-4 \
        --warmup-epochs 5 \
        --weight-decay {WEIGHT_DECAY[model_name]} \
        --clip-grad 0.02 \
        --save-freq 10 \
        --num-workers 8


## 5. Evaluate the best checkpoint

In [ ]:
MODEL = 'shvit_s4'
MODEL_DIR = f'{OUTPUT_DIR}/{MODEL}'

!python /content/finetune_shvit_ucf101.py \
    --model      {MODEL} \
    --shvit-dir  /content/SHViT \
    --finetune   "{MODEL_DIR}/best.pth" \
    --data-root  {DATA_ROOT} \
    --output-dir "{MODEL_DIR}" \
    --eval


In [ ]:
MODEL = 'shvit_s3'
MODEL_DIR = f'{OUTPUT_DIR}/{MODEL}'

!python /content/finetune_shvit_ucf101.py \
    --model      {MODEL} \
    --shvit-dir  /content/SHViT \
    --finetune   "{MODEL_DIR}/best.pth" \
    --data-root  {DATA_ROOT} \
    --output-dir "{MODEL_DIR}" \
    --eval


In [ ]:
MODEL = 'shvit_s2'
MODEL_DIR = f'{OUTPUT_DIR}/{MODEL}'

!python /content/finetune_shvit_ucf101.py \
    --model      {MODEL} \
    --shvit-dir  /content/SHViT \
    --finetune   "{MODEL_DIR}/best.pth" \
    --data-root  {DATA_ROOT} \
    --output-dir "{MODEL_DIR}" \
    --eval


In [ ]:
MODEL = 'shvit_s1'
MODEL_DIR = f'{OUTPUT_DIR}/{MODEL}'

!python /content/finetune_shvit_ucf101.py \
    --model      {MODEL} \
    --shvit-dir  /content/SHViT \
    --finetune   "{MODEL_DIR}/best.pth" \
    --data-root  {DATA_ROOT} \
    --output-dir "{MODEL_DIR}" \
    --eval


## 6. Plot training curves

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

MODELS = ['shvit_s1', 'shvit_s2', 'shvit_s3', 'shvit_s4']
COLORS = {'shvit_s1': 'tab:blue', 'shvit_s2': 'tab:orange',
          'shvit_s3': 'tab:green', 'shvit_s4': 'tab:red'}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

summary = []
for model_name in MODELS:
    csv_path = f'{OUTPUT_DIR}/{model_name}/training_log.csv'
    if not os.path.exists(csv_path):
        print(f'[skip] no log for {model_name}')
        continue

    df = pd.read_csv(csv_path)
    color = COLORS[model_name]

    axes[0].plot(df['epoch'], df['train_loss'], label=f'{model_name} train', color=color)
    axes[0].plot(df['epoch'], df['val_loss'],   label=f'{model_name} val',
                 color=color, linestyle='--', alpha=0.7)

    axes[1].plot(df['epoch'], df['val_top1'] * 100, label=f'{model_name} top-1', color=color)
    axes[1].plot(df['epoch'], df['val_top5'] * 100, label=f'{model_name} top-5',
                 color=color, linestyle='--', alpha=0.7)

    axes[2].plot(df['epoch'], df['lr'].astype(float), label=model_name, color=color)

    best = df.loc[df['val_top1'].idxmax()]
    summary.append((model_name, best['val_top1']*100, best['val_top5']*100, int(best['epoch'])))

axes[0].set_title('Loss');           axes[0].set_xlabel('epoch')
axes[0].legend(fontsize=8, ncol=2);  axes[0].grid(True)

axes[1].set_title('Val Accuracy (%)'); axes[1].set_xlabel('epoch')
axes[1].legend(fontsize=8, ncol=2);    axes[1].grid(True)

axes[2].set_title('Learning Rate');    axes[2].set_xlabel('epoch')
axes[2].set_yscale('log');             axes[2].legend(fontsize=8); axes[2].grid(True)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_curves_all.png', dpi=120)
plt.show()

print('\nBest val accuracy per model:')
print(f'{"model":<12} {"top-1":>8} {"top-5":>8} {"epoch":>6}')
for name, t1, t5, ep in summary:
    print(f'{name:<12} {t1:>7.2f}% {t5:>7.2f}% {ep:>6}')
